# Airbnb Athens Listing 

## Building a predictive model to forecast pricing 

## Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# from xgboost import XGBRegressor

## Data loading

In [2]:
listings = pd.read_csv('athens_listings.csv')
# reviews = pd.read_csv('reviews.csv')

## Data cleaning

### Checking for missing values

In [3]:
print(listings.isnull().sum())

# Fill missing values (if applicable)
listings['name'].fillna('No Name', inplace=True)
listings['reviews_per_month'].fillna(0, inplace=True)

id                                    0
name                                  0
host_id                               0
host_name                             0
neighbourhood_group               14137
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                               427
minimum_nights                        0
number_of_reviews                     0
last_review                        1858
reviews_per_month                  1858
calculated_host_listings_count        0
availability_365                      0
number_of_reviews_ltm                 0
license                             202
dtype: int64


C:\Users\κική\AppData\Local\Temp\ipykernel_3256\2751709751.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  listings['name'].fillna('No Name', inplace=True)
C:\Users\κική\AppData\Local\Temp\ipykernel_3256\2751709751.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

### Handling Duplicates

In [4]:
listings.drop_duplicates(inplace=True)

### Fixing Data Types

In [5]:
listings['last_review'] = pd.to_datetime(listings['last_review'], errors='coerce')

## Overall Summary

### How many listings are there?

In [6]:
print(f"Total listings: {listings.shape[0]}")

Total listings: 14137


In [7]:
listings = listings.dropna(subset=['last_review'])

## Defining the target and features

In [8]:
# Drop rows with missing prices
listings = listings.dropna(subset=['price']) 
listings['price'] = listings['price'].astype(float)

In [9]:
# Dropping columns
columns_to_drop = ['id', 'name', 'host_id', 'host_name', 'neighbourhood_group','license', 'last_review']
data_cleaned = listings.drop(columns=columns_to_drop)

In [10]:
# Interpolating 'lat' and 'long' based on 'neighbourhood'
# Grouping by 'neighbourhood' and calculating the mean 'lat' and 'long'
mean_coords = data_cleaned.groupby('neighbourhood')[['latitude', 'longitude']].mean()

# Applying the mean coordinates to missing values
data_cleaned = data_cleaned.set_index('neighbourhood')
data_cleaned['latitude'].fillna(mean_coords['latitude'], inplace=True)
data_cleaned['longitude'].fillna(mean_coords['longitude'], inplace=True)
data_cleaned.reset_index(inplace=True)

C:\Users\κική\AppData\Local\Temp\ipykernel_3256\1276043876.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_cleaned['latitude'].fillna(mean_coords['latitude'], inplace=True)
C:\Users\κική\AppData\Local\Temp\ipykernel_3256\1276043876.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a

In [11]:
# Filling columns about numerical data with median and mean
numerical_columns = ['minimum_nights', 'number_of_reviews',
                     'reviews_per_month', 'calculated_host_listings_count',
                     'availability_365']
for col in numerical_columns:
    data_cleaned[col].fillna(data_cleaned[col].median(), inplace=True)

C:\Users\κική\AppData\Local\Temp\ipykernel_3256\100482101.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_cleaned[col].fillna(data_cleaned[col].median(), inplace=True)
C:\Users\κική\AppData\Local\Temp\ipykernel_3256\100482101.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [12]:
# Filling columns about numerical data with mode
categorical_columns = ['room_type']
for col in categorical_columns:
    data_cleaned[col].fillna(data_cleaned[col].mode()[0], inplace=True)

C:\Users\κική\AppData\Local\Temp\ipykernel_3256\2510382449.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_cleaned[col].fillna(data_cleaned[col].mode()[0], inplace=True)


In [13]:
#'price_per_night' - price divided by minimum nights (if minimum nights is 0, set to 1 to avoid division by zero)
data_cleaned['minimum_nights'] = data_cleaned['minimum_nights'].replace(0, 1)
data_cleaned['price_per_night'] = data_cleaned['price'] / data_cleaned['minimum_nights'] 

## Feature engineering

In [14]:
# Applying one-hot encoding
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first') # drop='first' to avoid multicollinearity
encoded_data = pd.DataFrame(onehot_encoder.fit_transform(data_cleaned[categorical_columns]))
encoded_data.columns = onehot_encoder.get_feature_names_out(categorical_columns)

In [15]:
# Dropping original categorical columns and adding encoded columns
#data_fe variable created for feature engineering the cleaned data
data_fe = data_cleaned.drop(columns=categorical_columns)
data_fe = pd.concat([data_fe, encoded_data], axis=1)

In [16]:
non_num_columns = ['neighbourhood']
data_fe = data_fe.drop(columns=non_num_columns,axis=1)
data_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11954 entries, 0 to 11953
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   latitude                        11954 non-null  float64
 1   longitude                       11954 non-null  float64
 2   price                           11954 non-null  float64
 3   minimum_nights                  11954 non-null  int64  
 4   number_of_reviews               11954 non-null  int64  
 5   reviews_per_month               11954 non-null  float64
 6   calculated_host_listings_count  11954 non-null  int64  
 7   availability_365                11954 non-null  int64  
 8   number_of_reviews_ltm           11954 non-null  int64  
 9   price_per_night                 11954 non-null  float64
 10  room_type_Hotel room            11954 non-null  float64
 11  room_type_Private room          11954 non-null  float64
 12  room_type_Shared room           

In [17]:
data_fe.isnull().head(60)

,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,price_per_night,room_type_Hotel room,room_type_Private room,room_type_Shared room
0,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,False,False,False,False,False,False,False
8,False,False,False,False,False,False,False,False,False,False,False,False,False
9,False,False,False,False,False,False,False,False,False,False,False,False,False


## Train-test split

In [18]:
X = data_fe.iloc[:, :-1].values
y = data_fe.iloc[:, -1].values

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Buliding and training the model

#### Applying Linear Regression

In [20]:
# linear_reg = LinearRegression()
# linear_reg.fit(X_train, y_train)

#### Applying Decision Tree Regression

In [21]:
decision_tree_reg = DecisionTreeRegressor(random_state=101)
decision_tree_reg.fit(X_train, y_train)

DecisionTreeRegressor(random_state=101)

## Evaluate the model 

In [22]:
# y_pred_linear_reg = linear_reg.predict(X_test)

In [23]:
# mse_linear_reg = mean_squared_error(y_test, y_pred_linear_reg)
# r2_linear_reg = r2_score(y_test, y_pred_linear_reg)

# mse_linear_reg, r2_linear_reg
# (0.0035319561452903264, 0.0012006797969080774)

In [24]:
# Predicting on the test set
y_pred_decision_tree = decision_tree_reg.predict(X_test)

In [25]:
mse_decision_tree = mean_squared_error(y_test, y_pred_decision_tree)
r2_decision_tree = r2_score(y_test, y_pred_decision_tree)

# mse_decision_tree, r2_decision_tree

In [26]:
print(f"Mean Squared Error: {mse_decision_tree:.2f}")
print(f"R^2 Score: {r2_decision_tree:.2f}")

Mean Squared Error: 0.00
R^2 Score: 0.14
